# Safe Semi-Supervised Learning (SSL) - Starter Notebook
Safe SSL aims to leverage unlabeled data without degrading performance compared to the supervised baseline.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [ ]:
# Data split into labeled, unlabeled, and test
X, y = make_classification(n_samples=2500, n_features=18, n_informative=10, random_state=42)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_labeled, X_unlab, y_labeled, _ = train_test_split(X_temp, y_temp, test_size=0.75, random_state=42)

print('Labeled size :', len(X_labeled))
print('Unlabeled size:', len(X_unlab))
print('Test size    :', len(X_test))

In [ ]:
# Baseline supervised model
base = LogisticRegression(max_iter=1000, random_state=42)
base.fit(X_labeled, y_labeled)
base_pred = base.predict(X_test)
base_acc = accuracy_score(y_test, base_pred)
print(f'Baseline supervised accuracy: {base_acc:.4f}')

In [ ]:
# Safe pseudo-labeling: only add high-confidence unlabeled points
proba_unlab = base.predict_proba(X_unlab)
conf = proba_unlab.max(axis=1)
pseudo_y = proba_unlab.argmax(axis=1)

threshold = 0.95
safe_idx = conf >= threshold

X_aug = np.vstack([X_labeled, X_unlab[safe_idx]])
y_aug = np.hstack([y_labeled, pseudo_y[safe_idx]])

safe_model = LogisticRegression(max_iter=1000, random_state=42)
safe_model.fit(X_aug, y_aug)
safe_pred = safe_model.predict(X_test)
safe_acc = accuracy_score(y_test, safe_pred)

print('High-confidence pseudo labels used:', np.sum(safe_idx))
print(f'Safe SSL accuracy: {safe_acc:.4f}')
print(f'Gain over baseline: {safe_acc - base_acc:+.4f}')

In [ ]:
# Safety check logic
if safe_acc >= base_acc:
    print('Safe condition satisfied: SSL did not hurt baseline.')
else:
    print('Safety violated: revert to supervised baseline.')